# 2-Month Survival Prediction — starter notebook

Predict whether a critically ill patient is **`alive`** or **`dead`** at the 2-month
follow-up, from clinical data recorded at admission.

This notebook runs end-to-end: **load the data → train a simple model → submit**.
Metric: **accuracy**.

> Open in Colab, then run the cells top to bottom. The only thing you must edit is
> your API token in the next cell (find it on your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" scikit-learn pandas  # installs the `mlarena` package

import mlarena
import pandas as pd

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
COMPETITION_ID = 172

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Load the data

This session's data is small and lives next to this notebook in the workshop repo, so we read
it straight from GitHub — no download needed. (Sessions 2 & 3 instead pull their larger
image/text data through the SDK with `client.download_dataset(...)`.)

In [ ]:
BASE = "https://raw.githubusercontent.com/racousin/SCAI-4EUWorkshopAIinMedicineWorkshop/main/Hands-On-Session-1/data"

X_train = pd.read_csv(f"{BASE}/X_train.csv")
y_train = pd.read_csv(f"{BASE}/y_train.csv")
X_test  = pd.read_csv(f"{BASE}/X_test.csv")

print("X_train:", X_train.shape, "  X_test:", X_test.shape)
print(y_train["outcome"].value_counts())
X_train.head()

## 2. A baseline model

Impute missing values, one-hot the categorical columns, and fit a random forest.
This is just a starting point — replace it with your own features and models.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from pandas.api.types import is_numeric_dtype

ID = "patient_id"
features = [c for c in X_train.columns if c != ID]
num_cols = [c for c in features if is_numeric_dtype(X_train[c])]
cat_cols = [c for c in features if c not in num_cols]

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
])
model = Pipeline([("pre", pre),
                  ("clf", RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1))])

cv = cross_val_score(model, X_train[features], y_train["outcome"], cv=3, scoring="accuracy")
print("CV accuracy: %.4f +/- %.4f" % (cv.mean(), cv.std()))
model.fit(X_train[features], y_train["outcome"])

## 3. Predict and write `submission.csv`

In [ ]:
pred = model.predict(X_test[features])
submission = pd.DataFrame({"patient_id": X_test[ID], "outcome": pred})
submission.to_csv("submission.csv", index=False)
submission.head()

## 4. Submit

Run the cell below to submit through the SDK, or upload `submission.csv` on the
competition page.

In [ ]:
result = client.submit(competition_id=COMPETITION_ID, files=["submission.csv"])
print(result)

## 5. Where to go from here

The baseline is deliberately plain — beating it is the fun part. Work one step at a
time: **change one thing → re-check the CV score → submit only if it improved.** Trust the
cross-validated number, not a single lucky split.

- **EDA — visualize and understand the data.** Look before you model. Plot each feature's
  distribution, compare it across `alive` vs `dead`, and quantify missingness per column
  (`X_train.isna().mean()`). Watch for class imbalance, implausible values (negative ages,
  impossible vitals), and features that leak the outcome.

- **Data preprocessing.** This usually moves the score more than the model choice.
  - *Missing values:* try `median`/`most_frequent` vs `KNNImputer` or `IterativeImputer`;
    a binary "was-missing" flag is often informative in clinical data.
  - *Outliers:* clip or winsorize extreme values, and sanity-check units.
  - *Feature engineering:* ratios and interactions, binning continuous vitals into clinical
    ranges, and turning dates into durations (e.g. days since admission).

- **Model selection.** Go beyond the random forest: logistic regression (a strong, scaled
  linear baseline), SVM, and especially **gradient boosting — XGBoost / LightGBM /
  CatBoost**, which tend to win on tabular clinical data. Compare them under the *same* CV
  split so the numbers are honest.

- **Hyperparameter tuning.** Search instead of guessing — `GridSearchCV` /
  `RandomizedSearchCV`, or **Optuna** for larger spaces. Tune the few parameters that
  matter most (boosting: `n_estimators`, `learning_rate`, `max_depth`) and keep tuning
  inside cross-validation to avoid overfitting the validation set.

- **Stacking & ensembling.** Models that make *different* mistakes combine well. Average
  probabilities, or use `StackingClassifier` with a simple logistic-regression meta-model
  on top of your best estimators — this often edges out any single model.
